# 05c ML Fixed Embeddings — Group 3: Fine-Tuned (No-Log Target)

**Group 3.** Trains regressors on fine-tuned GNN embeddings. Target is the original scale `systemic_risk_label` (no log transform).

| Dataset | Model | Dim | Notes |
|---|---|---|---|
| `graphsage_fixed_32_srisk_nolog_dataset.parquet` | GraphSAGE v2 | 32 | Reconstruction loss; 32-dim |
| `graphsage_fixed_64_srisk_nolog_dataset.parquet` | GraphSAGE v3 | 64 | Reconstruction loss; 64-dim |
| `graphsage_fixed_128_srisk_nolog_dataset.parquet` | GraphSAGE v4 | 128 | Reconstruction loss; 128-dim |
| `node2vec_fixed_32_srisk_nolog_dataset.parquet` | Node2Vec v2 | 32 | Structural; 32-dim |
| `node2vec_fixed_64_srisk_nolog_dataset.parquet` | Node2Vec v3 | 64 | Structural; 64-dim |
| `node2vec_fixed_128_srisk_nolog_dataset.parquet` | Node2Vec v4 | 128 | Structural; 128-dim |

> Run `03c_embeddings_v3_nolog.ipynb` and `03d_embeddings_v4_128dim.ipynb` first to generate the parquet files.

In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

sys.path.insert(0, os.path.abspath('../..'))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]
TARGET_COL = "systemic_risk_label"
DISPLAY_COLS = [
    "dataset", "model",
    "train_mae", "validation_mae",
    "train_rmse", "validation_rmse",
]
TOP1_COLS = [
    "dataset", "model",
    "train_top1_mae", "validation_top1_mae",
    "train_top1_rmse", "validation_top1_rmse",
]

print(f"Project root: {PROJECT_ROOT}")

## Load No-Log Embedding Datasets

In [ ]:
DATASETS = {
    "GraphSAGE-v2": "graphsage_fixed_32_srisk_nolog_dataset.parquet",
    "GraphSAGE-v3": "graphsage_fixed_64_srisk_nolog_dataset.parquet",
    "GraphSAGE-v4": "graphsage_fixed_128_srisk_nolog_dataset.parquet",
    "Node2Vec-v2":  "node2vec_fixed_32_srisk_nolog_dataset.parquet",
    "Node2Vec-v3":  "node2vec_fixed_64_srisk_nolog_dataset.parquet",
    "Node2Vec-v4":  "node2vec_fixed_128_srisk_nolog_dataset.parquet",
}

loaded = {}
for name, filename in DATASETS.items():
    df, feature_cols = load_gnn_dataset(
        PROJECT_ROOT,
        target_col=TARGET_COL,
        filename=filename,
    )
    loaded[name] = (df, feature_cols)
    print(f"{name:14s} {df.shape}  embedding_cols={len(feature_cols)}")

In [ ]:
for name, (df, feature_cols) in loaded.items():
    print("
", name)
    print(df[["bank_id", "year", "quarter", "period", TARGET_COL]].head())

## Create Trainers

In [ ]:
trainers = {}
for name, (df, feature_cols) in loaded.items():
    trainers[name] = ModelTrainer(
        df=df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
    )
    trainer = trainers[name]
    print(f"{name:14s} train={trainer.train_df.shape} val={trainer.val_df.shape} test={trainer.test_df.shape}")

## Define Models

In [ ]:
candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

## Train All No-Log Experiments

In [ ]:
leaderboards = []
for dataset_name, trainer in trainers.items():
    trainer.train_all(candidate_models)
    board = trainer.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    leaderboards.append(board)
    print(f"\n--- {dataset_name} ---")
    display(board[DISPLAY_COLS])
    display(board[TOP1_COLS])

all_results = pd.concat(leaderboards, ignore_index=True)
print("\n=== All Results ===")
all_results[DISPLAY_COLS].sort_values(["validation_rmse", "validation_mae"]).reset_index(drop=True)

In [ ]:
top1_boards = []
for dataset_name, trainer in trainers.items():
    board = trainer.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    top1_boards.append(board)

pd.concat(top1_boards, ignore_index=True)[TOP1_COLS].sort_values(
    ["validation_top1_rmse", "validation_top1_mae"]
).reset_index(drop=True)

## Hyperparameter Tuning

In [ ]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300, 400, 500, 600],
    "model__max_depth":         [None, 5, 10, 15, 20, 30],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300, 400, 500, 600],
    "model__max_depth":         [3, 4, 5, 6, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400, 600, 800],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

tuned_boards = []
for dataset_name, t in trainers.items():
    print(f"\n--- Tuning {dataset_name} ---")
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, "MLP (tuned)")
    board = t.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    tuned_boards.append(board)

all_tuned = pd.concat(tuned_boards, ignore_index=True)
all_tuned[DISPLAY_COLS].sort_values(["validation_rmse", "validation_mae"]).reset_index(drop=True)

## Best Model Per Dataset

In [ ]:
best_rows = []
for dataset_name, trainer in trainers.items():
    board = trainer.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    best_rows.append(board.iloc[0])

best_results = pd.DataFrame(best_rows)
best_results[DISPLAY_COLS].sort_values(["validation_rmse", "validation_mae"]).reset_index(drop=True)